# 🏷️ Text classification: keywords vs meaning
### TF-IDF and sentence embeddings, head to head on the same advice-forum posts

**Module 2 · Session 01 · notebook 2 of 3**

A reader writes to an advice forum. Which section should the post go to? That is a **supervised** text
classification problem: we have posts with a known label, we learn from them, and we predict the label of
posts the model has not seen.

We build the same pipeline twice and change only one thing, the **representation** of the text:

1. **TF-IDF** (notebook 1, era 2): one dimension per word, a keyword fingerprint.
2. **Sentence embeddings** (era 4): 384 numbers per post from `all-MiniLM-L6-v2`, a meaning fingerprint.

The classifiers are the ones you know from module 1. Then we look past accuracy: which classes get confused,
what the model actually uses, where it goes wrong, and how many labelled posts each approach needs.

Run all cells (Runtime → Run all). Plotting code is folded away (*Show code*).

Contents
1. The data
2. Train/test split
3. Pipeline 1 · TF-IDF + linear models
4. Pipeline 2 · Sentence embeddings + the same models
5. Head to head
6. Which classes get confused?
7. What does the TF-IDF model look at?
8. Error analysis
9. How many labels do you need?
10. Takeaways

In [ ]:
# @title Setup: libraries and chart style { display-mode: "form" }
%pip install -q sentence-transformers
import re, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, sentence_transformers
from matplotlib.colors import LinearSegmentedColormap
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")

NAVY, LAV, GREY, ORANGE = "#211A52", "#C2C1CC", "#54616E", "#D2542A"
CMAP = LinearSegmentedColormap.from_list("aau", ["#FFFFFF", LAV, NAVY])
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": GREY, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E6E6EC", "legend.frameon": False})
pd.set_option("display.max_colwidth", 200)
print("scikit-learn", sklearn.__version__, "· sentence-transformers", sentence_transformers.__version__,
      "· pandas", pd.__version__)

---
# 📬 1 · The data

457 posts written in the style of a Danish advice column ("brevkasse"), in English. They are **synthetic**:
generated with an LLM and then checked, so no real person's problems end up in a classroom. Each post has a
title, a body (`selftext`) and one of eight labels.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/aaubs/ds-master/main/data/dk_posts_synth_en_processed.json"
df = pd.read_json(DATA_URL)
df["text"] = (df["title"] + " . " + df["selftext"]).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
print(df.shape)
df["label"].value_counts()

In [ ]:
df[["title", "label"]].sample(6, random_state=1)

🤔 *Before we model: which two labels do you expect to be hardest to tell apart, and why?*

---
# ✂️ 2 · Train/test split

Same rule as in module 1: the test posts stay locked away until the end. `stratify` keeps the share of each
label equal in both parts.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.25, random_state=42, stratify=df["label"])
print(f"train: {len(X_train)} posts · test: {len(X_test)} posts · {y_train.nunique()} labels")
print(f"baseline (always guess the most common label): {y_test.value_counts(normalize=True).max():.0%}")

---
# 📊 3 · Pipeline 1 · TF-IDF + linear models

`TfidfVectorizer` turns each post into a sparse vector, one dimension per word or word pair
(`ngram_range=(1, 2)` adds pairs like *new job*). We try two linear classifiers: **logistic regression** and a
**linear support vector machine**. Both learn one weight per word and label.

A `Pipeline` keeps vectorizer and classifier together, so the vocabulary is learned from the training posts only.

In [ ]:
tfidf_lr  = make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 2)),
                          LogisticRegression(max_iter=1000, random_state=42))
tfidf_svm = make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 2)), LinearSVC(random_state=42))

results = {}
for name, model in [("TF-IDF + logistic regression", tfidf_lr), ("TF-IDF + linear SVM", tfidf_svm)]:
    model.fit(X_train, y_train)
    results[name] = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:<32} accuracy {results[name]:.3f}")

print(f"\nvocabulary: {len(tfidf_lr[0].vocabulary_):,} words and word pairs")

---
# 🧠 4 · Pipeline 2 · Sentence embeddings + the same models

Now each post becomes 384 numbers from `all-MiniLM-L6-v2`, the model from notebook 1. We compute the
embeddings once (this takes up to a minute on a CPU) and train exactly the same two classifiers on them.

The embedding model was trained by someone else on a billion sentence pairs. We do not change it; we only use
it to *represent* our posts. That is **transfer learning**: knowledge about language, learned elsewhere,
reused for our task.

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")
E_train = encoder.encode(X_train.tolist(), show_progress_bar=True)
E_test = encoder.encode(X_test.tolist())
print("embeddings:", E_train.shape)

In [ ]:
emb_lr  = LogisticRegression(max_iter=1000, random_state=42).fit(E_train, y_train)
emb_svm = LinearSVC(random_state=42).fit(E_train, y_train)

for name, model in [("Embeddings + logistic regression", emb_lr), ("Embeddings + linear SVM", emb_svm)]:
    results[name] = accuracy_score(y_test, model.predict(E_test))
    print(f"{name:<32} accuracy {results[name]:.3f}")

---
# 🥊 5 · Head to head

In [ ]:
# @title Chart: accuracy of the four pipelines { display-mode: "form" }
res = pd.Series(results).sort_values()
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(res.index, res.values, color=[NAVY if n.startswith("Emb") else LAV for n in res.index])
for i, v in enumerate(res.values):
    ax.text(v + 0.005, i, f"{v:.2f}", va="center", color=GREY)
ax.set_xlim(0, 1); ax.set_xlabel("accuracy on 115 test posts"); ax.grid(axis="y", visible=False)
plt.show()

Both representations beat the 14 % baseline by far, and the gap between them is small. With 115 test posts, one
post is almost one percentage point, so a ranking between pipelines that are a few points apart can change
with another random split or another library version.

The question to ask is not "which number is highest" but "what does each one get wrong, and what does it cost
me to run it".

---
# 🔀 6 · Which classes get confused?

The confusion matrix from module 1, now with eight classes. Rows are the true label, columns the prediction;
everything off the diagonal is a mistake.

In [ ]:
# @title Confusion matrices: TF-IDF vs embeddings (logistic regression) { display-mode: "form" }
classes = sorted(y_test.unique())
short = ["Rants", "Family", "Friendship", "Health", "Love", "Money & housing", "Practical", "Work"]  # same order
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (title, pred) in zip(axes, [("TF-IDF + logistic regression", tfidf_lr.predict(X_test)),
                                    ("Embeddings + logistic regression", emb_lr.predict(E_test))]):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=classes, display_labels=short, ax=ax,
                                            cmap=CMAP, colorbar=False, xticks_rotation=45)
    ax.set_title(title, color=NAVY); ax.grid(False)
axes[1].set_ylabel(""); axes[1].set_yticklabels([])
plt.tight_layout(); plt.show()

In [ ]:
print(classification_report(y_test, emb_lr.predict(E_test)))

🤔 *Which pair of labels is confused most? Was it the pair you guessed in section 1? Read two posts from that
pair: would you, as a human, always agree on the label?*

---
# 🔍 7 · What does the TF-IDF model look at?

A big advantage of TF-IDF + logistic regression: every word has a weight per label, so we can read the model.
Embedding dimensions have no such meaning.

In [ ]:
vec, clf = tfidf_lr[0], tfidf_lr[-1]
words = vec.get_feature_names_out()
top = {label: [words[j] for j in np.argsort(clf.coef_[i])[::-1][:8]] for i, label in enumerate(clf.classes_)}
pd.DataFrame(top).T.rename(columns=lambda r: f"#{r+1}")

🤔 *Do these words make sense as signals for each label? Is any of them a word that would not carry over to
real posts (a name, a phrase the generator liked)?*

---
# 🩺 8 · Error analysis

Accuracy hides the interesting part. Below are the test posts the TF-IDF model got wrong with the **highest
confidence**: confident mistakes are the dangerous ones in a real system.

In [ ]:
proba = tfidf_lr.predict_proba(X_test)
errors = pd.DataFrame({"text": X_test.values, "true": y_test.values,
                       "predicted": tfidf_lr.predict(X_test), "confidence": proba.max(axis=1),
                       "embeddings say": emb_lr.predict(E_test)})
errors = errors[errors["true"] != errors["predicted"]].sort_values("confidence", ascending=False)
print(f"{len(errors)} of {len(X_test)} test posts misclassified by TF-IDF + logistic regression")
errors.head(6)

### ✏️ Try it: ask an AI, then check
Pick one of the posts above. Paste the post, its true label and the predicted label into any chat assistant
and ask: *"Why might a keyword-based classifier have put this post in the wrong category?"*

Then check the explanation against the evidence in this notebook:
* Are the words it blames actually among the top words for the predicted label (section 7)?
* Is the confusion it describes one that shows up in the confusion matrix (section 6)?
* Does it claim the true label is obviously right? Read the post yourself: is it?

An AI explanation is a hypothesis. The model's weights and errors are the evidence.

---
# 📉 9 · How many labels do you need?

Labelling posts costs time and money. So a fair question is: how does each approach do when it has only a
handful of labelled examples? We train both on 16, 25, 50, 100, 200 and all 342 training posts, five random
draws each, and always test on the same 115 posts.

In [ ]:
rows = []
y_tr = y_train.to_numpy()
for n in [16, 25, 50, 100, 200, len(X_train)]:
    for seed in range(5):
        idx = (np.arange(len(X_train)) if n == len(X_train) else
               train_test_split(np.arange(len(X_train)), train_size=n, stratify=y_tr, random_state=seed)[0])
        m1 = make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 2)),
                           LogisticRegression(max_iter=1000)).fit(X_train.iloc[idx], y_tr[idx])
        m2 = LogisticRegression(max_iter=1000).fit(E_train[idx], y_tr[idx])
        rows += [{"n": n, "method": "TF-IDF", "accuracy": accuracy_score(y_test, m1.predict(X_test))},
                 {"n": n, "method": "Embeddings", "accuracy": accuracy_score(y_test, m2.predict(E_test))}]
curve = pd.DataFrame(rows).groupby(["method", "n"])["accuracy"].agg(["mean", "std"]).reset_index()
curve.pivot(index="n", columns="method", values="mean").round(2)

In [ ]:
# @title Chart: accuracy by number of labelled posts { display-mode: "form" }
fig, ax = plt.subplots(figsize=(8, 4.5))
for method, color in [("Embeddings", NAVY), ("TF-IDF", ORANGE)]:
    c = curve[curve["method"] == method]
    ax.plot(c["n"], c["mean"], marker="o", color=color, label=method, linewidth=2)
    ax.fill_between(c["n"], c["mean"] - c["std"], c["mean"] + c["std"], color=color, alpha=0.15)
ax.set_xscale("log"); ax.set_xticks([], minor=True)
ax.set_xticks([16, 25, 50, 100, 200, 342]); ax.set_xticklabels([16, 25, 50, 100, 200, 342])
ax.set_xlabel("labelled training posts (log scale)"); ax.set_ylabel("test accuracy"); ax.set_ylim(0, 1)
ax.legend(); plt.show()

This is where the representation really matters. With few labels, TF-IDF has only seen a few words per class
and is stuck; the embeddings already "know" that *lonely* and *making friends* belong together, because the
encoder learned that elsewhere. As labels pile up, TF-IDF catches up. The left end of the curve is bumpy:
with 16 or 25 posts, *which* posts you happen to draw matters a lot (the shaded band).

In practice: if labelling is expensive, start from embeddings. If you have thousands of labels and need speed
and explainability, TF-IDF is hard to beat.

---
# 🧰 10 · Takeaways

| | TF-IDF + linear model | Embeddings + linear model |
|---|---|---|
| Accuracy with all labels | close | close |
| Accuracy with few labels | weak | much better |
| Speed and cost | very fast, runs anywhere | needs an encoder pass (a GPU helps at scale) |
| Explainability | read the word weights | dimensions have no meaning |
| Synonyms and paraphrase | no | yes |

**Start with the simple baseline, then earn the complexity.** Next, notebook 3 drops the labels altogether:
can we discover the themes in these posts without being told what they are?